In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import pandas as pd
from sklearn import metrics
import config_test
import numpy as np

In [2]:
device = "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential( #forward pass computation of the input data through the layers in the order they appear.
            nn.Linear(784, 512),
            nn.ReLU(), #activation function
            nn.Linear(512, 512),
            nn.ReLU(), #activation function
            nn.Linear(512, 10),
        )

    def forward(self, x): #going through one layer to another
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
# Load and initialize data
df = pd.read_csv(config_test.NEW_DATA_PATH)
dt = torch.tensor(df.values, dtype=torch.float32)
#print(dt)

# Test single model on different folds
for fold in range(config_test.FOLDS_COUNT):
    #initialize

    neural_network_torch = NeuralNetwork().to(device) #model
    
    df_train = df.loc[df["kfold"] != fold, :].reset_index(drop=True)
    
    df_test = df.loc[df["kfold"] == fold, :].reset_index(drop=True)

    dt_train = torch.tensor(df_train.values, dtype=torch.float32)
    dt_train_label = dt_train[:,-2:-1]
    dt_train = dt_train[:,:-2]
    dt_train_label = dt_train_label.long()

    dt_test = torch.tensor(df_test.values, dtype=torch.float32)
    dt_test_label = dt_test[:,-2:-1]
    dt_test = dt_test[:,:-2]
    dt_test_label = dt_test_label.long()

    train_dataset = TensorDataset(dt_train, dt_train_label)
    train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)

    test_dataset = TensorDataset(dt_test, dt_test_label)
    test_loader = DataLoader(test_dataset, batch_size=200, shuffle=False)

    criterion = nn.CrossEntropyLoss()  # Funkcja kosztu dla klasyfikacji
    optimizer = torch.optim.Adam(neural_network_torch.parameters(), lr=0.001)  # Optymalizator

    #training loop
    torch.manual_seed(42)

    epochs = 10

    # Put data to target device
    dt_train, dt_train_label = dt_train.to(device), dt_train_label.to(device)
    dt_test, dt_test_label = dt_test.to(device), dt_test_label.to(device)

    # training loop
    for epoch in range(epochs):

        neural_network_torch.train()

        running_loss = 0.0

        for X_batch, y_batch in train_loader: #X_batch - kawałek dt_drain, y_batch - kawałek dt_train_label
            logits = neural_network_torch(X_batch)
            y_batch = y_batch.squeeze(1)
            loss = criterion(logits, y_batch)

            # 3. Optimizer zero grad (czyszczenie poprzednich gradientów)
            optimizer.zero_grad()

            # 4. backpropagation
            loss.backward()

            # 5. aktualizacja wag
            optimizer.step()

            running_loss += loss.item()
        
        #testing
        neural_network_torch.eval()  # <- wyłączamy dropouty, batch norm itp.
        correct = 0
        total = 0
        val_loss = 0

        with torch.no_grad():  # <- bardzo ważne: nie liczymy gradientów podczas testu!
            for X_val, y_val in test_loader:
                logits = neural_network_torch(X_val)
                y_val = y_val.squeeze(1)
                loss = criterion(logits, y_val)
                val_loss += loss.item()

                pred = logits.argmax(dim=1)
                correct += (pred == y_val).sum().item()
                total += y_val.size(0)

        accuracy = correct / total

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}, Accuracy: {accuracy}")



Epoch 1/10, Loss: 3.674288149923086, Accuracy: 0.898
Epoch 2/10, Loss: 0.1886443704366684, Accuracy: 0.9325
Epoch 3/10, Loss: 0.09294514367356896, Accuracy: 0.944
Epoch 4/10, Loss: 0.042012983560562135, Accuracy: 0.946
Epoch 5/10, Loss: 0.020513003738597036, Accuracy: 0.9475
Epoch 6/10, Loss: 0.014698162931017578, Accuracy: 0.9485
Epoch 7/10, Loss: 0.008612092200201005, Accuracy: 0.949
Epoch 8/10, Loss: 0.004962243890622631, Accuracy: 0.9505
Epoch 9/10, Loss: 0.0024366224330151453, Accuracy: 0.9515
Epoch 10/10, Loss: 0.0013103689590934664, Accuracy: 0.951
Epoch 1/10, Loss: 3.8338431529700756, Accuracy: 0.8985
Epoch 2/10, Loss: 0.20350761357694863, Accuracy: 0.9335
Epoch 3/10, Loss: 0.08910965183749794, Accuracy: 0.941
Epoch 4/10, Loss: 0.04051147997379303, Accuracy: 0.9405
Epoch 5/10, Loss: 0.017571409093216063, Accuracy: 0.949
Epoch 6/10, Loss: 0.007068862958112732, Accuracy: 0.951
Epoch 7/10, Loss: 0.003317070109187625, Accuracy: 0.9485
Epoch 8/10, Loss: 0.0018490766684408299, Accura